# Huawei-Inspired Healthcare Machine Learning Lab 1
## Predicting Hospital Length of Stay with Linear Regression

**Purpose:** Adapt the step-by-step linear regression workflow in the Huawei HCIA-AI V4.0 Machine Learning Lab Guide to a healthcare dataset.

**Educational use only:** This notebook is not a clinical decision-support system and must not be used to diagnose, treat, or manage patients.

### Official dataset
- **UCI Diabetes 130-US Hospitals for Years 1999–2008**
- Dataset page: https://archive.ics.uci.edu/dataset/296/diabetes-130-us-hospitals-for-years-1999-2008
- DOI: https://doi.org/10.24432/C5230J
- Licence: CC BY 4.0

The dataset contains hospital encounters of patients diagnosed with diabetes. In this lab, `time_in_hospital` is used as the numeric outcome to demonstrate regression.
## How to read this notebook
- A line starting with `#` is a comment for you. Python does not run it.
- Run one code cell at a time from top to bottom.
- The comments explain what the computer is doing in simple healthcare language.


## Learning objectives

By the end of this lab, learners should be able to:

1. Load a public healthcare dataset.
2. Select numeric input features and a continuous target.
3. Split data into training and testing sets.
4. Train a linear regression model.
5. Evaluate the model using MAE, RMSE, and R².
6. Interpret predictions and model coefficients responsibly.

## Step 1 — Install and import the required packages

The `ucimlrepo` package downloads the dataset directly from the official UCI Machine Learning Repository.

In [ ]:
# Install the small UCI helper package so Colab can download the dataset for us.
!pip -q install ucimlrepo

# NumPy helps us work with numbers and arrays.
import numpy as np
# Pandas helps us work with tables, similar to an Excel sheet.
import pandas as pd
# Matplotlib helps us draw charts.
import matplotlib.pyplot as plt

# This function downloads a dataset from the official UCI repository.
from ucimlrepo import fetch_ucirepo
# This function divides the data into a training part and a testing part.
from sklearn.model_selection import train_test_split
# This fills in missing numerical values using the middle value, called the median.
from sklearn.impute import SimpleImputer
# A pipeline keeps the data-preparation and prediction steps together.
from sklearn.pipeline import Pipeline
# Linear regression is the machine-learning model used in this lab.
from sklearn.linear_model import LinearRegression
# These tools measure how close the predictions are to the real values.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Using the same random-state number makes our results repeatable.
RANDOM_STATE = 42
print('Packages imported successfully.')

## Step 2 — Download the healthcare dataset

UCI dataset ID **296** is fetched directly into the notebook.

In [ ]:
# Download the UCI healthcare dataset using its official dataset number.
dataset = fetch_ucirepo(id=296)

# Copy the input table into a DataFrame.
# A DataFrame is a table with rows and columns, similar to a spreadsheet.
df = dataset.data.features.copy()

# Show basic information so we know the dataset loaded correctly.
print('Dataset name:', dataset.metadata.name)
print('Number of rows:', df.shape[0])
print('Number of columns:', df.shape[1])
# Show the first five patient encounters.
df.head()

## Step 3 — Select the input features and target

### Target
`time_in_hospital`: number of days between admission and discharge.

### Input features
This beginner lab uses numeric count variables so that the regression workflow remains easy to understand:

- `num_lab_procedures`
- `num_procedures`
- `num_medications`
- `number_outpatient`
- `number_emergency`
- `number_inpatient`
- `number_diagnoses`

These variables are used for teaching only. Their inclusion does not establish clinical causation.

In [ ]:
# This is the value we want the model to predict.
# Here, it is the number of days the patient stayed in hospital.
target_column = 'time_in_hospital'

# These are the pieces of information the model will use to make its prediction.
feature_columns = [
    'num_lab_procedures',      # Number of laboratory tests
    'num_procedures',          # Number of procedures
    'num_medications',         # Number of medications recorded
    'number_outpatient',       # Previous outpatient visits
    'number_emergency',        # Previous emergency visits
    'number_inpatient',        # Previous inpatient admissions
    'number_diagnoses'         # Number of diagnoses recorded
]

# Check that all the columns we need are really present in the dataset.
missing_columns = [
    column for column in feature_columns + [target_column]
    if column not in df.columns
]

# Stop the notebook with a clear message if an expected column is missing.
if missing_columns:
    raise ValueError(f'Missing expected columns: {missing_columns}')

# Keep only the columns needed for this lab.
model_data = df[feature_columns + [target_column]].copy()
# Convert the selected values into numbers. Invalid entries become missing values.
model_data = model_data.apply(pd.to_numeric, errors='coerce')
# Remove rows where the answer we want to predict is missing.
model_data = model_data.dropna(subset=[target_column])

# Show the size and first five rows of the final modelling table.
print('Modelling dataset shape:', model_data.shape)
model_data.head()

## Step 4 — Explore the target variable

In [ ]:
# Display a simple statistical summary of hospital stay duration.
# This includes the average, minimum, maximum and quartiles.
print(model_data[target_column].describe())

# Create a histogram to show how often each length of stay occurs.
plt.figure(figsize=(8, 5))
plt.hist(model_data[target_column], bins=14, edgecolor='black')
plt.xlabel('Time in hospital (days)')
plt.ylabel('Number of encounters')
plt.title('Distribution of Hospital Length of Stay')
plt.show()

## Step 5 — Split the data

The model learns from the training set and is evaluated on a separate test set.

In [ ]:
# X contains the information the model uses to learn.
X = model_data[feature_columns]
# y contains the correct hospital-stay value for each encounter.
y = model_data[target_column]

# Keep 80% of the data for learning and 20% for testing.
# The model does not see the test answers during training.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

# Confirm how many examples are in each part.
print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## Step 6 — Build and train the linear regression model

A median imputer is included so the code remains robust if a selected numeric feature contains missing values.
**In simple words:** We first repair missing values, then ask the model to learn how the selected hospital variables are related to length of stay.


In [ ]:
# Build a two-step workflow.
# Step 1 fills in missing numbers using the median.
# Step 2 trains the linear regression model.
model = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('regressor', LinearRegression())
])

# Ask the model to learn patterns from the training data.
model.fit(X_train, y_train)
print('Model training completed.')

## Step 7 — Make predictions

In [ ]:
# Use the trained model to estimate hospital stay for the test encounters.
y_pred = model.predict(X_test)

# Place the first ten real and predicted values side by side.
prediction_preview = pd.DataFrame({
    'Actual stay': y_test.values[:10],
    'Predicted stay': np.round(y_pred[:10], 2)
})

prediction_preview

## Step 8 — Evaluate the model

- **MAE:** average absolute prediction error in days.
- **RMSE:** gives more weight to larger errors.
- **R²:** proportion of variation explained by this simple linear model.

A modest or low R² is not automatically a coding error. Hospital length of stay is influenced by many clinical, organisational, and social factors that are not fully represented by these seven variables.
**Important:** A lower MAE and RMSE are better. R² can be low even when the code is correct because real hospital stays depend on many factors that are not included here.


In [ ]:
# MAE tells us the average size of the error in days.
mae = mean_absolute_error(y_test, y_pred)
# RMSE also measures error, but gives extra weight to large mistakes.
rmse = mean_squared_error(y_test, y_pred) ** 0.5
# R-squared tells us how much of the variation the model explains.
r2 = r2_score(y_test, y_pred)

print(f'Mean Absolute Error: {mae:.3f} days')
print(f'Root Mean Squared Error: {rmse:.3f} days')
print(f'R-squared: {r2:.3f}')

## Step 9 — Visualise actual versus predicted values

In [ ]:
# Draw one point for every test encounter.
# The horizontal position is the real value and the vertical position is the prediction.
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.25)

# Create a diagonal reference line.
# Points close to this line are more accurate predictions.
minimum_value = min(y_test.min(), y_pred.min())
maximum_value = max(y_test.max(), y_pred.max())
plt.plot([minimum_value, maximum_value], [minimum_value, maximum_value])

plt.xlabel('Actual length of stay (days)')
plt.ylabel('Predicted length of stay (days)')
plt.title('Actual versus Predicted Length of Stay')
plt.show()

## Step 10 — Inspect the model coefficients

The sign of a coefficient shows the direction of the model's statistical association while other included variables are held constant. It must not be interpreted as proof that the feature causes a longer or shorter stay.

In [ ]:
# Take the trained regression part out of the pipeline.
linear_regression = model.named_steps['regressor']

# Put each feature and its coefficient into a clear table.
# A positive coefficient means the model links higher values with a longer stay.
# A negative coefficient means the model links higher values with a shorter stay.
# This is association only, not proof of cause.
coefficient_table = pd.DataFrame({
    'Feature': feature_columns,
    'Coefficient': linear_regression.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('Intercept:', round(linear_regression.intercept_, 4))
coefficient_table

## Step 11 — Predict for one educational example

The values below are a fictional example and do not represent a real patient.

In [ ]:
# Create one fictional encounter using the same columns used during training.
example_encounter = pd.DataFrame([{
    'num_lab_procedures': 45,
    'num_procedures': 1,
    'num_medications': 12,
    'number_outpatient': 0,
    'number_emergency': 1,
    'number_inpatient': 0,
    'number_diagnoses': 7
}])

# Ask the model to estimate the hospital stay for this fictional example.
example_prediction = model.predict(example_encounter)[0]
print(f'Estimated length of stay: {example_prediction:.2f} days')

## Step 12 — Compare with a simple baseline

A useful model should be compared with a basic strategy. Here, the baseline predicts the average training-set length of stay for every test encounter.

In [ ]:
# MAE tells us the average size of the error in days.
mae = mean_absolute_error(y_test, y_pred)
# RMSE also measures error, but gives extra weight to large mistakes.
rmse = mean_squared_error(y_test, y_pred) ** 0.5
# R-squared tells us how much of the variation the model explains.
r2 = r2_score(y_test, y_pred)

print(f'Mean Absolute Error: {mae:.3f} days')
print(f'Root Mean Squared Error: {rmse:.3f} days')
print(f'R-squared: {r2:.3f}')

## Student exercises

1. Remove one feature and check whether the results change.
2. Add another suitable numeric feature from the dataset.
3. Change the train/test split from 80/20 to 70/30.
4. Plot residuals: `actual - predicted`.
5. Compare linear regression with Ridge regression.
6. Explain why prediction error may remain high even when the code is correct.

## Responsible-use checklist

- Use only de-identified, legally accessible data.
- Do not enter identifiable patient information into this notebook.
- Report test-set performance, not training performance alone.
- Do not describe association as causation.
- Do not deploy the model clinically without external validation, governance, ethics review, and appropriate regulatory assessment.

## Dataset citation

Clore, J., Cios, K., DeShazo, J., & Strack, B. (2014). *Diabetes 130-US Hospitals for Years 1999–2008* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5230J